# 01 — Data Preparation
Load and flatten the raw Steam datasets, then write the interim data.

In [8]:
from pathlib import Path

# Paths to the two raw Steam datasets.
# Keeping these in one place makes the rest of the notebook easier to read.
RAW_DIR = Path("../data/raw")

USER_ITEMS_PATH = RAW_DIR / "australian_users_items.json.gz"
GAMES_PATH = RAW_DIR / "steam_games.json.gz"

print("User items file exists:", USER_ITEMS_PATH.exists())
print("Games metadata file exists:", GAMES_PATH.exists())

User items file exists: True
Games metadata file exists: True


In [9]:
import ast
import gzip


def read_first_record(file_path):
    with gzip.open(file_path, "rt", encoding="utf-8") as file:
        first_line = file.readline()

    return ast.literal_eval(first_line)

# Inspect one raw record from each dataset before doing any processing.
user_record = read_first_record(USER_ITEMS_PATH)
game_record = read_first_record(GAMES_PATH)

print("USER RECORD:")
print(user_record)

print()

print("GAME RECORD:")
print(game_record)

USER RECORD:
{'user_id': '76561197970982479', 'items_count': 277, 'steam_id': '76561197970982479', 'user_url': 'http://steamcommunity.com/profiles/76561197970982479', 'items': [{'item_id': '10', 'item_name': 'Counter-Strike', 'playtime_forever': 6, 'playtime_2weeks': 0}, {'item_id': '20', 'item_name': 'Team Fortress Classic', 'playtime_forever': 0, 'playtime_2weeks': 0}, {'item_id': '30', 'item_name': 'Day of Defeat', 'playtime_forever': 7, 'playtime_2weeks': 0}, {'item_id': '40', 'item_name': 'Deathmatch Classic', 'playtime_forever': 0, 'playtime_2weeks': 0}, {'item_id': '50', 'item_name': 'Half-Life: Opposing Force', 'playtime_forever': 0, 'playtime_2weeks': 0}, {'item_id': '60', 'item_name': 'Ricochet', 'playtime_forever': 0, 'playtime_2weeks': 0}, {'item_id': '70', 'item_name': 'Half-Life', 'playtime_forever': 0, 'playtime_2weeks': 0}, {'item_id': '130', 'item_name': 'Half-Life: Blue Shift', 'playtime_forever': 0, 'playtime_2weeks': 0}, {'item_id': '300', 'item_name': 'Day of Defea

## Inspect the raw data

Before processing the datasets, we inspect a small sample to understand their structure and decide which fields are relevant.

In [10]:
# Show the structure of one user without printing the full list of games.
print("USER FIELDS:")
print(user_record.keys())

print("\nUSER SUMMARY:")
print("User ID:", user_record["user_id"])
print("Steam ID:", user_record["steam_id"])
print("Number of items:", user_record["items_count"])

print("\nFIRST 3 ITEMS:")
for item in user_record["items"][:3]:
    print(item)


print("\nGAME METADATA FIELDS:")
print(game_record.keys())

print("\nGAME EXAMPLE:")
print(game_record)

USER FIELDS:
dict_keys(['user_id', 'items_count', 'steam_id', 'user_url', 'items'])

USER SUMMARY:
User ID: 76561197970982479
Steam ID: 76561197970982479
Number of items: 277

FIRST 3 ITEMS:
{'item_id': '10', 'item_name': 'Counter-Strike', 'playtime_forever': 6, 'playtime_2weeks': 0}
{'item_id': '20', 'item_name': 'Team Fortress Classic', 'playtime_forever': 0, 'playtime_2weeks': 0}
{'item_id': '30', 'item_name': 'Day of Defeat', 'playtime_forever': 7, 'playtime_2weeks': 0}

GAME METADATA FIELDS:
dict_keys(['publisher', 'genres', 'app_name', 'title', 'url', 'release_date', 'tags', 'discount_price', 'reviews_url', 'specs', 'price', 'early_access', 'id', 'developer'])

GAME EXAMPLE:
{'publisher': 'Kotoshiro', 'genres': ['Action', 'Casual', 'Indie', 'Simulation', 'Strategy'], 'app_name': 'Lost Summoner Kitty', 'title': 'Lost Summoner Kitty', 'url': 'http://store.steampowered.com/app/761140/Lost_Summoner_Kitty/', 'release_date': '2018-01-04', 'tags': ['Strategy', 'Action', 'Indie', 'Casual

## Flatten one user record

The user-item data is nested: each user contains a list of games.
For later analysis, we want one row per user-game interaction.

In [11]:
import pandas as pd


# Convert the first user's game list into one row per user-game interaction.
sample_rows = []

for item in user_record["items"][:3]:
    sample_rows.append({
        "user_id": user_record["user_id"],
        "steam_id": user_record["steam_id"],
        "game_id": item["item_id"],
        "game_name": item["item_name"],
        "playtime_forever": item["playtime_forever"],
        "playtime_2weeks": item["playtime_2weeks"]
    })

sample_df = pd.DataFrame(sample_rows)

sample_df

,user_id,steam_id,game_id,game_name,playtime_forever,playtime_2weeks
0,76561197970982479,76561197970982479,10,Counter-Strike,6,0
1,76561197970982479,76561197970982479,20,Team Fortress Classic,0,0
2,76561197970982479,76561197970982479,30,Day of Defeat,7,0


## Flatten a user record

Each user contains a nested list of games. We convert this into one row per user-game interaction so the data can later be stored in a normal table.

In [12]:
def flatten_user(user):
    """Convert one user's nested game list into tabular rows."""
    rows = []

    for item in user["items"]:
        rows.append({
            "user_id": user["user_id"],
            "steam_id": user["steam_id"],
            "game_id": item["item_id"],
            "game_name": item["item_name"],
            "playtime_forever": item["playtime_forever"],
            "playtime_2weeks": item["playtime_2weeks"]
        })

    return rows


user_rows = flatten_user(user_record)

print("Number of rows:", len(user_rows))
pd.DataFrame(user_rows).head()

Number of rows: 277


,user_id,steam_id,game_id,game_name,playtime_forever,playtime_2weeks
0,76561197970982479,76561197970982479,10,Counter-Strike,6,0
1,76561197970982479,76561197970982479,20,Team Fortress Classic,0,0
2,76561197970982479,76561197970982479,30,Day of Defeat,7,0
3,76561197970982479,76561197970982479,40,Deathmatch Classic,0,0
4,76561197970982479,76561197970982479,50,Half-Life: Opposing Force,0,0


## Flatten all user-game interactions

The raw file stores games inside each user record. We convert this into one row per user-game interaction and save the result as a Parquet file.

The data is written in batches to avoid keeping the entire dataset in memory.

In [13]:
import pyarrow as pa
import pyarrow.parquet as pq


INTERIM_DIR = Path("../data/interim")
OUTPUT_PATH = INTERIM_DIR / "interactions_flat.parquet"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)


def flatten_all_users(input_path, output_path, batch_size=100_000):
    """Flatten all user-game interactions and save them as Parquet."""
    rows = []
    writer = None

    user_count = 0
    interaction_count = 0

    with gzip.open(input_path, "rt", encoding="utf-8") as file:
        for line in file:
            user = ast.literal_eval(line)
            user_count += 1

            for item in user["items"]:
                rows.append({
                    "user_id": user["user_id"],
                    "steam_id": user["steam_id"],
                    "game_id": item["item_id"],
                    "game_name": item["item_name"],
                    "playtime_forever": item["playtime_forever"],
                    "playtime_2weeks": item["playtime_2weeks"]
                })

            # Write a batch once enough rows have been collected.
            if len(rows) >= batch_size:
                table = pa.Table.from_pandas(pd.DataFrame(rows), preserve_index=False)

                if writer is None:
                    writer = pq.ParquetWriter(output_path, table.schema)

                writer.write_table(table)

                interaction_count += len(rows)
                rows = []

    # Write any remaining rows.
    if rows:
        table = pa.Table.from_pandas(pd.DataFrame(rows), preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema)

        writer.write_table(table)
        interaction_count += len(rows)

    if writer is not None:
        writer.close()

    print("Users processed:", user_count)
    print("Interactions written:", interaction_count)
    print("Saved to:", output_path)


flatten_all_users(USER_ITEMS_PATH, OUTPUT_PATH)

Users processed: 88310
Interactions written: 5153209
Saved to: ../data/interim/interactions_flat.parquet


## Verify the flattened dataset

We load the saved Parquet file again to confirm that the conversion worked correctly before moving on to cleaning.

In [14]:
interactions = pd.read_parquet(OUTPUT_PATH)

print("Shape:", interactions.shape)

interactions.tail()

Shape: (5153209, 6)


,user_id,steam_id,game_id,game_name,playtime_forever,playtime_2weeks
5153204,76561198329548331,76561198329548331,346330,BrainBread 2,0,0
5153205,76561198329548331,76561198329548331,373330,All Is Dust,0,0
5153206,76561198329548331,76561198329548331,388490,One Way To Die: Steam Edition,3,3
5153207,76561198329548331,76561198329548331,521570,You Have 10 Seconds 2,4,4
5153208,76561198329548331,76561198329548331,519140,Minds Eyes,3,3


## Check data types and missing values

Before cleaning the data, we check the column types and whether any values are missing.

In [15]:
print("DATA TYPES:")
print(interactions.dtypes)

print("\nMISSING VALUES:")
print(interactions.isna().sum())

DATA TYPES:
user_id               str
steam_id              str
game_id               str
game_name             str
playtime_forever    int64
playtime_2weeks     int64
dtype: object

MISSING VALUES:
user_id             0
steam_id            0
game_id             0
game_name           0
playtime_forever    0
playtime_2weeks     0
dtype: int64


## Check playtime values

Playtime should never be negative, so we check for invalid values before continuing with the cleaning.

In [16]:
negative_forever = (interactions["playtime_forever"] < 0).sum()
negative_2weeks = (interactions["playtime_2weeks"] < 0).sum()

print("Negative lifetime playtime:", negative_forever)
print("Negative 2-week playtime:", negative_2weeks)

Negative lifetime playtime: 0
Negative 2-week playtime: 0


## Check duplicate user-game interactions

Each user-game pair should ideally appear once. We check whether the same user is linked to the same game multiple times.

In [17]:
duplicate_pairs = interactions.duplicated(
    subset=["user_id", "game_id"],
    keep=False
)

print("Rows involved in duplicate user-game pairs:", duplicate_pairs.sum())

Rows involved in duplicate user-game pairs: 117774


## Inspect duplicate user-game pairs

Some users have the same game recorded more than once. Before removing anything, we inspect these rows to see whether the duplicates are identical or contain different playtime values.

In [18]:
duplicate_rows = interactions[duplicate_pairs].sort_values(
    ["user_id", "game_id"]
)

duplicate_rows.head(20)

,user_id,steam_id,game_id,game_name,playtime_forever,playtime_2weeks
1094154,05041129,76561198167088451,104900,ORION: Prelude,0,0
2712151,05041129,76561198167088451,104900,ORION: Prelude,0,0
1094152,05041129,76561198167088451,105600,Terraria,650,0
2712149,05041129,76561198167088451,105600,Terraria,650,0
1094156,05041129,76561198167088451,17080,Tribes: Ascend,0,0
2712153,05041129,76561198167088451,17080,Tribes: Ascend,0,0
1094155,05041129,76561198167088451,204360,Castle Crashers,223,0
2712152,05041129,76561198167088451,204360,Castle Crashers,223,0
1094159,05041129,76561198167088451,218230,PlanetSide 2,19,0
2712156,05041129,76561198167088451,218230,PlanetSide 2,19,0


In [19]:
# Group all duplicated rows by user and game.
duplicate_groups = duplicate_rows.groupby(["user_id", "game_id"])

# For each duplicate group, count how many different values appear
# in the remaining columns.
value_counts = duplicate_groups[
    ["steam_id", "game_name", "playtime_forever", "playtime_2weeks"]
].nunique()

# A pair is conflicting if at least one column has more than one unique value.
conflicting_pairs = value_counts.gt(1).any(axis=1)

print("Duplicate user-game pairs:", duplicate_groups.ngroups)
print("Exact duplicate pairs:", (~conflicting_pairs).sum())
print("Conflicting duplicate pairs:", conflicting_pairs.sum())

Duplicate user-game pairs: 58647
Exact duplicate pairs: 58625
Conflicting duplicate pairs: 22


In [20]:
# Get the user-game pairs where duplicate rows disagree on at least one value.
conflicting_index = conflicting_pairs[conflicting_pairs].index

conflicting_rows = duplicate_rows.set_index(
    ["user_id", "game_id"]
).loc[conflicting_index].reset_index()

conflicting_rows

,user_id,game_id,steam_id,game_name,playtime_forever,playtime_2weeks
0,76561198050680344,377160,76561198050680344,Fallout 4,1997,1997
1,76561198050680344,377160,76561198050680344,Fallout 4,2058,2058
2,76561198064484479,232090,76561198064484479,Killing Floor 2,2231,170
3,76561198064484479,232090,76561198064484479,Killing Floor 2,2231,116
4,76561198064956087,40100,76561198064956087,Supreme Commander 2,2504,60
5,76561198064956087,40100,76561198064956087,Supreme Commander 2,2543,99
6,76561198072861800,433850,76561198072861800,H1Z1: King of the Kill,5083,142
7,76561198072861800,433850,76561198072861800,H1Z1: King of the Kill,5083,111
8,76561198072861800,433850,76561198072861800,H1Z1: King of the Kill,5084,112
9,76561198079079942,282070,76561198079079942,This War of Mine,1486,1486


In [21]:
# Count how many duplicate groups have conflicting values in each column.
conflicts_by_column = (value_counts > 1).sum()

print(conflicts_by_column)

steam_id             0
game_name            0
playtime_forever    10
playtime_2weeks     22
dtype: int64


## Remove duplicate user-game interactions

Most duplicate user-game pairs are exact copies. A small number contain different playtime values.

Since lifetime playtime is cumulative, we keep the record with the highest `playtime_forever` for each user-game pair.

In [22]:
# Sort so the highest lifetime playtime appears first for each user-game pair.
interactions_clean = interactions.sort_values(
    "playtime_forever",
    ascending=False
)

# Keep one row per user-game pair.
interactions_clean = interactions_clean.drop_duplicates(
    subset=["user_id", "game_id"],
    keep="first"
)

print("Rows before:", len(interactions))
print("Rows after:", len(interactions_clean))
print("Rows removed:", len(interactions) - len(interactions_clean))

Rows before: 5153209
Rows after: 5094082
Rows removed: 59127


In [23]:
remaining_duplicates = interactions_clean.duplicated(
    subset=["user_id", "game_id"]
).sum()

print("Remaining duplicate user-game pairs:", remaining_duplicates)

Remaining duplicate user-game pairs: 0


## Check game ID consistency

A game ID should ideally correspond to one game name. We check whether any IDs are associated with multiple names.

In [24]:
# Count how many different names are associated with each game ID.
names_per_game = interactions_clean.groupby("game_id")["game_name"].nunique()

games_with_multiple_names = (names_per_game > 1).sum()

print("Game IDs with multiple names:", games_with_multiple_names)

Game IDs with multiple names: 0


## Check user ID consistency

Each `user_id` should ideally correspond to one `steam_id`. We check whether any users are associated with multiple Steam IDs.

In [25]:
# Count how many different Steam IDs are associated with each user ID.
steam_ids_per_user = interactions_clean.groupby("user_id")["steam_id"].nunique()

users_with_multiple_steam_ids = (steam_ids_per_user > 1).sum()

print("User IDs with multiple Steam IDs:", users_with_multiple_steam_ids)

User IDs with multiple Steam IDs: 0


In [26]:
# Count how many different user IDs are associated with each Steam ID.
user_ids_per_steam_id = interactions_clean.groupby("steam_id")["user_id"].nunique()

steam_ids_with_multiple_users = (user_ids_per_steam_id > 1).sum()

print("Steam IDs with multiple user IDs:", steam_ids_with_multiple_users)

Steam IDs with multiple user IDs: 0


## Save cleaned interactions

The duplicate user-game records have now been resolved, so we save the cleaned interaction table for the next stages of the project.

In [27]:
CLEAN_INTERACTIONS_PATH = Path("../data/processed/interactions_clean.parquet")

# steam_id is redundant because it maps one-to-one with user_id.
interactions_clean = interactions_clean.drop(columns=["steam_id"])

interactions_clean.to_parquet(
    CLEAN_INTERACTIONS_PATH,
    index=False
)

print("Saved cleaned interactions to:")
print(CLEAN_INTERACTIONS_PATH)
print("Shape:", interactions_clean.shape)

Saved cleaned interactions to:
../data/processed/interactions_clean.parquet
Shape: (5094082, 5)


## Prepare game metadata

We extract the relevant game information from the raw metadata file and save it as a clean Parquet table for later analysis.

In [28]:
# Keep only the metadata fields that are useful for our project.
game_columns = [
    "id",
    "title",
    "genres",
    "tags",
    "developer",
    "publisher",
    "release_date",
    "price",
    "early_access"
]

game_rows = []

with gzip.open(GAMES_PATH, "rt", encoding="utf-8") as file:
    for line in file:
        game = ast.literal_eval(line)

        game_rows.append({
            column: game.get(column)
            for column in game_columns
        })

games = pd.DataFrame(game_rows)

# Rename the ID column to match the interaction dataset.
games = games.rename(columns={"id": "game_id"})

print("Shape:", games.shape)
games.head()

Shape: (32135, 9)


,game_id,title,genres,tags,developer,publisher,release_date,price,early_access
0,761140,Lost Summoner Kitty,"[Action, Casual, Indie, Simulation, Strategy]","[Strategy, Action, Indie, Casual, Simulation]",Kotoshiro,Kotoshiro,2018-01-04,4.99,False
1,643980,Ironbound,"[Free to Play, Indie, RPG, Strategy]","[Free to Play, Strategy, Indie, RPG, Card Game...",Secret Level SRL,"Making Fun, Inc.",2018-01-04,Free To Play,False
2,670290,Real Pool 3D - Poolians,"[Casual, Free to Play, Indie, Simulation, Sports]","[Free to Play, Simulation, Sports, Casual, Ind...",Poolians.com,Poolians.com,2017-07-24,Free to Play,False
3,767400,弹炸人2222,"[Action, Adventure, Casual]","[Action, Adventure, Casual]",彼岸领域,彼岸领域,2017-12-07,0.99,False
4,773570,NaN,None,"[Action, Indie, Casual, Sports]",NaN,NaN,NaN,2.99,False


In [29]:
# Convert numeric prices normally.
# Any text-based price labels are treated as 0.0 since price is not used in our analysis.
games["price"] = pd.to_numeric(
    games["price"],
    errors="coerce"
).fillna(0.0)

# Remove rows without a game ID and keep one row per game.
games_clean = (
    games
    .dropna(subset=["game_id"])
    .drop_duplicates(subset=["game_id"])
    .reset_index(drop=True)
)

GAMES_CLEAN_PATH = Path("../data/processed/games_clean.parquet")

games_clean.to_parquet(
    GAMES_CLEAN_PATH,
    index=False
)

print("Saved game metadata to:")
print(GAMES_CLEAN_PATH)
print("Shape:", games_clean.shape)

Saved game metadata to:
../data/processed/games_clean.parquet
Shape: (32132, 9)


## Check metadata coverage

We compare the game IDs in the interaction data with the metadata to see how many games can be matched between the two datasets.

In [30]:
interaction_game_ids = set(interactions_clean["game_id"])
metadata_game_ids = set(games_clean["game_id"])

matched_games = interaction_game_ids & metadata_game_ids
missing_metadata = interaction_game_ids - metadata_game_ids

print("Games in interactions:", len(interaction_game_ids))
print("Games with metadata:", len(matched_games))
print("Games missing metadata:", len(missing_metadata))

Games in interactions: 10978
Games with metadata: 9192
Games missing metadata: 1786


## Data preparation summary

The raw Steam data has now been converted into two cleaned datasets:

- `interactions_clean.parquet`: 5,094,082 unique user-game interactions
- `games_clean.parquet`: metadata for 32,132 unique games

The interaction data contains 10,978 unique games, of which 9,192 have matching metadata. Games without metadata are kept because the behavioural analysis does not depend on metadata.